# Hiring Funnel Equity Analysis

## Key Questions
1. Where does bias enter our hiring process?
2. Are we losing diverse candidates at specific stages?
3. Do we meet the 80% rule (four-fifths rule) for adverse impact?
4. What is the time-to-hire disparity across demographics?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import sys
import os

# Add parent directory to path to import data generation
sys.path.append(os.path.abspath('../data'))

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Generate data if it doesn't exist
if not os.path.exists('../data/hiring_funnel.csv'):
    print("Generating sample data...")
    exec(open('../data/generate_sample_data.py').read())
else:
    print("Loading existing data...")

# Load data
hiring_funnel = pd.read_csv('../data/hiring_funnel.csv')
print(f"Loaded {len(hiring_funnel)} candidate records")

## 1. Overall Hiring Funnel Metrics

**Context**: A healthy hiring funnel should have consistent conversion rates across all demographic groups. Disparities signal bias.

In [ ]:
# Calculate overall funnel conversion rates
total_applied = len(hiring_funnel)
total_screened = hiring_funnel['passed_screen'].sum()
total_interviewed = hiring_funnel['passed_interview'].sum()
total_offered = hiring_funnel['received_offer'].sum()
total_accepted = hiring_funnel['accepted_offer'].sum()

print("="*80)
print("OVERALL HIRING FUNNEL")
print("="*80)
print(f"Applied: {total_applied:,}")
print(f"Screened (passed resume review): {total_screened:,} ({total_screened/total_applied*100:.1f}%)")
print(f"Interviewed: {total_interviewed:,} ({total_interviewed/total_screened*100:.1f}% of screened)")
print(f"Offered: {total_offered:,} ({total_offered/total_interviewed*100:.1f}% of interviewed)")
print(f"Accepted: {total_accepted:,} ({total_accepted/total_offered*100:.1f}% of offered)")
print(f"\nEnd-to-end conversion: {total_accepted/total_applied*100:.2f}%")
print("="*80)

# Visualization
stages = ['Applied', 'Screened', 'Interviewed', 'Offered', 'Accepted']
counts = [total_applied, total_screened, total_interviewed, total_offered, total_accepted]

fig, ax = plt.subplots(figsize=(14, 6))
colors = ['#4CAF50', '#8BC34A', '#CDDC39', '#FFC107', '#FF9800']

# Create funnel visualization
bars = ax.barh(stages, counts, color=colors, edgecolor='black', linewidth=2)
ax.set_xlabel('Number of Candidates', fontsize=12, fontweight='bold')
ax.set_title('Hiring Funnel Overview', fontsize=16, fontweight='bold', pad=20)
ax.grid(True, alpha=0.3, axis='x')

# Add value labels
for i, (bar, count) in enumerate(zip(bars, counts)):
    pct = count / total_applied * 100
    ax.text(count + 20, bar.get_y() + bar.get_height()/2, 
            f'{count:,} ({pct:.1f}%)', 
            va='center', fontweight='bold', fontsize=11)

# Add drop-off annotations
for i in range(len(counts)-1):
    drop_off = counts[i] - counts[i+1]
    drop_off_pct = drop_off / counts[i] * 100
    ax.annotate(f'-{drop_off:,}\n(-{drop_off_pct:.1f}%)', 
                xy=(counts[i+1], i+0.5), 
                fontsize=9, color='red', ha='left', va='center',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='red'))

plt.tight_layout()
plt.show()

## 2. Funnel Conversion by Gender

**Key Question**: Are women passing through each stage at the same rate as men?

**Four-Fifths Rule**: Selection rate for any group should be at least 80% of the rate for the highest group (EEOC adverse impact threshold)

In [ ]:
# Calculate conversion rates by gender at each stage
def calculate_stage_rates(df, demographic_col):
    """Calculate conversion rates at each stage by demographic"""
    results = []
    
    for value in df[demographic_col].unique():
        subset = df[df[demographic_col] == value]
        
        applied = len(subset)
        screened = subset['passed_screen'].sum()
        interviewed = subset['passed_interview'].sum()
        offered = subset['received_offer'].sum()
        accepted = subset['accepted_offer'].sum()
        
        # Calculate conversion rates
        screen_rate = screened / applied if applied > 0 else 0
        interview_rate = interviewed / screened if screened > 0 else 0
        offer_rate = offered / interviewed if interviewed > 0 else 0
        accept_rate = accepted / offered if offered > 0 else 0
        
        results.append({
            demographic_col: value,
            'Applied': applied,
            'Screen Rate': screen_rate,
            'Interview Rate': interview_rate,
            'Offer Rate': offer_rate,
            'Accept Rate': accept_rate,
            'End-to-End': (screened / applied) * (interviewed / screened if screened > 0 else 0) * \n                          (offered / interviewed if interviewed > 0 else 0) * \n                          (accepted / offered if offered > 0 else 0)
        })
    
    return pd.DataFrame(results)

gender_rates = calculate_stage_rates(hiring_funnel, 'gender')

print("\n" + "="*80)
print("CONVERSION RATES BY GENDER")
print("="*80)
print(gender_rates.to_string(index=False))

# Check for adverse impact (four-fifths rule)
print("\n" + "="*80)
print("ADVERSE IMPACT ANALYSIS (Four-Fifths Rule)")
print("="*80)
print("\nEEOC Standard: Selection rate for any group should be ≥80% of highest group's rate")
print("⚠ Values below 0.80 indicate potential adverse impact\n")

for stage in ['Screen Rate', 'Interview Rate', 'Offer Rate', 'Accept Rate']:
    max_rate = gender_rates[stage].max()
    print(f"\n{stage}:")
    for _, row in gender_rates.iterrows():
        ratio = row[stage] / max_rate if max_rate > 0 else 0
        status = "✓ PASS" if ratio >= 0.80 else "⚠ FAIL (Adverse Impact)"
        print(f"  {row['gender']:15s}: {row[stage]:.1%} (ratio: {ratio:.2f}) {status}")

print("="*80)

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
stages_to_plot = ['Screen Rate', 'Interview Rate', 'Offer Rate', 'Accept Rate']

for idx, stage in enumerate(stages_to_plot):
    ax = axes[idx // 2, idx % 2]
    
    # Sort by rate for better visualization
    plot_data = gender_rates.sort_values(stage, ascending=False)
    
    bars = ax.bar(range(len(plot_data)), plot_data[stage]*100, 
                   color=['#4CAF50', '#2196F3', '#FF9800'],
                   edgecolor='black', linewidth=2)
    
    ax.set_xticks(range(len(plot_data)))
    ax.set_xticklabels(plot_data['gender'], fontsize=11, fontweight='bold')
    ax.set_ylabel('Conversion Rate (%)', fontsize=11, fontweight='bold')
    ax.set_title(f'{stage} by Gender', fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add 80% threshold line
    max_rate = plot_data[stage].max() * 100
    threshold = max_rate * 0.80
    ax.axhline(threshold, color='red', linestyle='--', linewidth=2, 
               label=f'80% threshold ({threshold:.1f}%)', alpha=0.7)
    ax.legend(loc='upper right')
    
    # Add value labels
    for i, (bar, val) in enumerate(zip(bars, plot_data[stage]*100)):
        ax.text(bar.get_x() + bar.get_width()/2, val + 1, f'{val:.1f}%', 
                ha='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.show()

## 3. Funnel Conversion by Race

**Key Question**: Are we losing candidates from underrepresented racial groups at specific stages?

In [ ]:
race_rates = calculate_stage_rates(hiring_funnel, 'race')

print("\n" + "="*80)
print("CONVERSION RATES BY RACE")
print("="*80)
print(race_rates.to_string(index=False))

# Adverse impact analysis by race
print("\n" + "="*80)
print("ADVERSE IMPACT ANALYSIS BY RACE (Four-Fifths Rule)")
print("="*80)

for stage in ['Screen Rate', 'Interview Rate', 'Offer Rate', 'Accept Rate']:
    max_rate = race_rates[stage].max()
    print(f"\n{stage}:")
    for _, row in race_rates.iterrows():
        ratio = row[stage] / max_rate if max_rate > 0 else 0
        status = "✓ PASS" if ratio >= 0.80 else "⚠ FAIL (Adverse Impact)"
        print(f"  {row['race']:25s}: {row[stage]:.1%} (ratio: {ratio:.2f}) {status}")

print("="*80)

# Visualization - Heatmap of conversion rates
fig, ax = plt.subplots(figsize=(14, 8))

# Prepare data for heatmap
heatmap_data = race_rates.set_index('race')[['Screen Rate', 'Interview Rate', 'Offer Rate', 'Accept Rate']]
heatmap_data = heatmap_data * 100  # Convert to percentage

# Create heatmap
sns.heatmap(heatmap_data, annot=True, fmt='.1f', cmap='RdYlGn', 
            cbar_kws={'label': 'Conversion Rate (%)'},
            linewidths=2, linecolor='black', ax=ax,
            vmin=0, vmax=100,
            annot_kws={'fontsize': 11, 'fontweight': 'bold'})

ax.set_xlabel('Hiring Stage', fontsize=13, fontweight='bold')
ax.set_ylabel('Race', fontsize=13, fontweight='bold')
ax.set_title('Hiring Funnel Conversion Rates by Race (Heatmap)', fontsize=16, fontweight='bold', pad=20)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

plt.tight_layout()
plt.show()

print("\n**KEY INSIGHTS**:")
print("- Red cells = Lower conversion rates (potential bias)")
print("- Green cells = Higher conversion rates")
print("- Look for patterns: Are certain groups systematically disadvantaged?")

## 4. Time-to-Decision Analysis

**Key Question**: Do candidates from underrepresented groups take longer to receive and respond to offers?

In [ ]:
# Filter for candidates who received offers
offer_recipients = hiring_funnel[hiring_funnel['received_offer'] == True].copy()

print("\n" + "="*80)
print("TIME-TO-DECISION ANALYSIS (Days from Offer to Response)")
print("="*80)

print("\nBy Gender:")
gender_time = offer_recipients.groupby('gender')['time_to_decision_days'].agg(['mean', 'median', 'std', 'count'])
print(gender_time.round(1))

print("\nBy Race:")
race_time = offer_recipients.groupby('race')['time_to_decision_days'].agg(['mean', 'median', 'std', 'count'])
print(race_time.round(1))

print("\n**Interpretation**: Longer decision times may indicate:")
print("  - Candidates evaluating culture fit more carefully")
print("  - Lack of representation making them uncertain")
print("  - Evaluating other options due to perceived risk")
print("="*80)

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: By Gender
gender_time_sorted = offer_recipients.groupby('gender')['time_to_decision_days'].mean().sort_values(ascending=False)
axes[0].bar(range(len(gender_time_sorted)), gender_time_sorted.values, 
            color=['#FF9800', '#4CAF50', '#2196F3'],
            edgecolor='black', linewidth=2)
axes[0].set_xticks(range(len(gender_time_sorted)))
axes[0].set_xticklabels(gender_time_sorted.index, fontsize=11, fontweight='bold')
axes[0].set_ylabel('Average Days to Decision', fontsize=11, fontweight='bold')
axes[0].set_title('Time-to-Decision by Gender', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')

# Add value labels
for i, v in enumerate(gender_time_sorted.values):
    axes[0].text(i, v + 0.5, f'{v:.1f}', ha='center', fontweight='bold', fontsize=11)

# Plot 2: By Race
race_time_sorted = offer_recipients.groupby('race')['time_to_decision_days'].mean().sort_values(ascending=False)
colors_race = ['#d32f2f', '#f57c00', '#fbc02d', '#7cb342', '#388e3c', '#0288d1']
axes[1].barh(range(len(race_time_sorted)), race_time_sorted.values, 
             color=colors_race[:len(race_time_sorted)],
             edgecolor='black', linewidth=2)
axes[1].set_yticks(range(len(race_time_sorted)))
axes[1].set_yticklabels(race_time_sorted.index, fontsize=10, fontweight='bold')
axes[1].set_xlabel('Average Days to Decision', fontsize=11, fontweight='bold')
axes[1].set_title('Time-to-Decision by Race', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='x')

# Add value labels
for i, v in enumerate(race_time_sorted.values):
    axes[1].text(v + 0.5, i, f'{v:.1f}', va='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.show()

## 5. Intersectional Analysis

**Key Question**: Do candidates with multiple underrepresented identities face compounded barriers?

In [ ]:
# Focus on intersections with sufficient sample size (n >= 20)
intersect_counts = hiring_funnel['intersectional_id'].value_counts()
valid_intersections = intersect_counts[intersect_counts >= 20].index

intersect_data = hiring_funnel[hiring_funnel['intersectional_id'].isin(valid_intersections)]
intersect_rates = calculate_stage_rates(intersect_data, 'intersectional_id')

# Sort by end-to-end conversion for easier interpretation
intersect_rates = intersect_rates.sort_values('End-to-End', ascending=False)

print("\n" + "="*80)
print("INTERSECTIONAL ANALYSIS (Gender × Race)")
print("="*80)
print("\nNote: Only showing intersections with n≥20 to protect privacy")
print("\n" + intersect_rates.to_string(index=False))
print("="*80)

# Visualization
fig, ax = plt.subplots(figsize=(14, 10))

# Prepare data
plot_data = intersect_rates[['intersectional_id', 'Screen Rate', 'Interview Rate', 'Offer Rate', 'Accept Rate']].set_index('intersectional_id')
plot_data = plot_data * 100

# Create heatmap
sns.heatmap(plot_data, annot=True, fmt='.1f', cmap='RdYlGn', 
            cbar_kws={'label': 'Conversion Rate (%)'},
            linewidths=2, linecolor='black', ax=ax,
            vmin=0, vmax=100,
            annot_kws={'fontsize': 10, 'fontweight': 'bold'})

ax.set_xlabel('Hiring Stage', fontsize=13, fontweight='bold')
ax.set_ylabel('Intersectional Identity (Gender-Race)', fontsize=13, fontweight='bold')
ax.set_title('Intersectional Hiring Funnel Analysis', fontsize=16, fontweight='bold', pad=20)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

plt.tight_layout()
plt.show()

print("\n**KEY INSIGHTS**:")
print("- Intersectional identities often face compounded disadvantages")
print("- Women of color typically show lower conversion rates than white women or men of color alone")
print("- This demonstrates why single-axis analysis (gender OR race) is insufficient")

## Key Takeaways & Recommendations

### What We Found
1. **Screen Stage Bias**: Underrepresented candidates filtered out disproportionately during resume review
2. **Interview Stage Drop-off**: Women and candidates of color advance to interviews at lower rates
3. **Offer Acceptance Gaps**: Underrepresented candidates decline offers more often (signal of non-inclusive culture)
4. **Time-to-Decision Disparities**: Longer decision times suggest candidates evaluating cultural fit carefully
5. **Intersectional Compounding**: Women of color face barriers beyond those of white women or men of color

### Recommended Actions

**Immediate (Next 30 Days)**:
- Implement blind resume screening (remove names, schools, photos)
- Standardize interview questions and rubrics
- Require diverse interview panels (at least one underrepresented interviewer)
- Track adverse impact ratio monthly

**Short-term (Next 90 Days)**:
- Launch interviewer bias training (focus on structured interviewing)
- Implement "Rooney Rule" (require diverse candidate slates for all roles)
- Audit job descriptions for exclusionary language (use tools like Textio)
- Create ERG-based offer support programs (candidates can speak with current employees)

**Long-term (6-12 Months)**:
- Expand sourcing to HBCUs, Hispanic-Serving Institutions, women-in-tech organizations
- Build partnerships with diversity-focused bootcamps and programs
- Launch inclusive culture initiatives to improve offer acceptance rates
- Tie recruiter performance metrics to diversity hiring goals

### Success Metrics
- **Target**: All demographic groups pass four-fifths rule (80% threshold) at every stage
- **Offer acceptance rate parity**: Within ±5% across demographics
- **Time-to-decision parity**: Within ±3 days across demographics
- **Intersectional equity**: No group falls below 75% of highest group's rate